# 极坐标 SNO：Operator Learner 训练

本 Notebook 只训练 Operator Learner（OL），不重新训练 Function Encoder（FE）。训练过程为：

\[
(z_f,\,g_n,\,k) \longmapsto z_P \longmapsto P(r,	heta).
\]

主要特性：

- 从已保存的 `fe_params.msgpack` 恢复并冻结 FE；
- OL 每一步都从 PI-sampler 分布生成新的训练 batch，不使用固定 pool；
- 使用 `tqdm` 显示训练进度；
- 定期在独立新样本上记录潜空间 MSE、潜空间相对 L2 和物理空间相对 L2；
- 持续保存 CSV、NPZ 和最新 OL checkpoint。

## 1. GPU 环境

以下单元必须在导入 JAX 前运行。如果当前内核已经导入过 JAX，请先重启内核。`GPU_ID` 使用重映射前的服务器 GPU 编号。

In [1]:
import os

GPU_ID = "4"  # 按服务器实际情况修改
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# 如需限制显存比例，可取消下一行注释：
# os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.85"

print("CUDA_VISIBLE_DEVICES =", os.environ["CUDA_VISIBLE_DEVICES"])

CUDA_VISIBLE_DEVICES = 4


## 2. 导入项目代码

服务器环境需安装 `requirements.txt` 中的依赖，尤其是新增的 `tqdm`：

```bash
pip install -r requirements.txt
```

In [2]:
from pathlib import Path
from dataclasses import fields, replace
import json
import sys

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from flax import serialization

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "config_polar.py").exists():
    PROJECT_DIR = Path("/home/user/data/Hollon/海洋工程水动力/polar_annulus_sno_code")

if not (PROJECT_DIR / "config_polar.py").exists():
    raise FileNotFoundError("PROJECT_DIR 中找不到 config_polar.py，请修改项目路径。")

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from config_polar import PolarAnnulusConfig
from data_polar import sample_batch
from train_polar import (
    create_fe_state,
    load_normalizer,
    train_operator,
)

print("Project directory:", PROJECT_DIR)
print("JAX devices:", jax.devices())

Project directory: /home/user/data/Hollon/海洋工程水动力/polar_annulus_sno_code
JAX devices: [CudaDevice(id=0)]


## 3. 从 FE 训练目录恢复配置

直接读取 FE 训练时保存的 `config.json`，避免手动复制网络维度造成 checkpoint 结构不匹配。旧配置若缺少新的 OL 日志字段，会自动采用默认值。

In [3]:
FE_RUN_NAME = "polar_v1"
OUTPUT_ROOT = PROJECT_DIR / "out_polar_annulus_sno"
fe_run_dir = OUTPUT_ROOT / FE_RUN_NAME
config_path = fe_run_dir / "config.json"

if not config_path.exists():
    raise FileNotFoundError(f"找不到 FE 配置：{config_path}")

saved_config = json.loads(config_path.read_text(encoding="utf-8"))
valid_config_fields = {field.name for field in fields(PolarAnnulusConfig)}
unknown_fields = sorted(set(saved_config) - valid_config_fields)
if unknown_fields:
    print("Ignoring unknown config fields:", unknown_fields)

config_kwargs = {
    key: value
    for key, value in saved_config.items()
    if key in valid_config_fields
}
config_kwargs["out_dir"] = str(OUTPUT_ROOT)
config_kwargs["run_name"] = FE_RUN_NAME

# 兼容重构前保存的旧 config.json。
config_kwargs.setdefault("ol_log_interval", 500)
config_kwargs.setdefault("ol_checkpoint_interval", 10_000)
config_kwargs.setdefault("ol_eval_sample_size", 16)
config_kwargs.setdefault("ol_eval_probe_points", 256)

cfg = PolarAnnulusConfig(**config_kwargs)

print("FE run directory:", fe_run_dir)
print("Effective full batch size:", cfg.effective_batch_size)
print("OL steps in saved config:", cfg.ol_steps)
print("OL log interval:", cfg.ol_log_interval)
print("OL checkpoint interval:", cfg.ol_checkpoint_interval)
print("OL eval sample size per scale:", cfg.ol_eval_sample_size)
print("OL eval probe points:", cfg.ol_eval_probe_points)

FE run directory: /home/user/data/Hollon/海洋工程水动力/polar_annulus_sno_code/out_polar_annulus_sno/polar_v1
Effective full batch size: 768
OL steps in saved config: 500000
OL log interval: 500
OL checkpoint interval: 10000
OL eval sample size per scale: 16
OL eval probe points: 256


## 4. 选择 smoke 或 full 模式

第一次运行必须先用 `smoke`。确认进度条、日志和 checkpoint 正常产生后，再切换为 `full`。

- `smoke`：使用相同网络结构和 FE 权重，但只采一个先验尺度、每步 2 个样本、训练 5 步，并输出到独立目录。
- `full`：使用 FE 训练时的完整采样分布和正式训练步数。

In [4]:
OL_RUN_MODE = "full"  # "smoke" 或 "full"

if OL_RUN_MODE == "smoke":
    ol_cfg = replace(
        cfg,
        prior_scale_pairs=(cfg.prior_scale_pairs[0],),
        repeats_per_scale=1,
        sample_size=2,
        ol_steps=5,
        ol_log_interval=1,
        ol_checkpoint_interval=5,
        ol_eval_sample_size=2,
        ol_eval_probe_points=4,
        run_name=f"{cfg.run_name}_ol_smoke",
    )
else:
    ol_cfg = replace(
        cfg,
        sample_size = 128,
        ol_steps=500_000,
        ol_log_interval=500,
        ol_checkpoint_interval=10_000,
        ol_eval_sample_size=32,
        ol_eval_probe_points=1024,
    )

print("OL mode:", OL_RUN_MODE)
print("OL output directory:", ol_cfg.output_dir)
print("OL effective batch size:", ol_cfg.effective_batch_size)
print("OL steps:", ol_cfg.ol_steps)
print("Log every:", ol_cfg.ol_log_interval, "steps")
print("Checkpoint every:", ol_cfg.ol_checkpoint_interval, "steps")
print("Eval sample size per scale:", ol_cfg.ol_eval_sample_size)
print("Eval probe points:", ol_cfg.ol_eval_probe_points)

OL mode: full
OL output directory: /home/user/data/Hollon/海洋工程水动力/polar_annulus_sno_code/out_polar_annulus_sno/polar_v1
OL effective batch size: 384
OL steps: 500000
Log every: 500 steps
Checkpoint every: 10000 steps
Eval sample size per scale: 32
Eval probe points: 1024


## 5. 加载并冻结 Function Encoder

FE 权重和归一化统计量始终从正式 FE 目录读取。`smoke` 只改变采样批量和 OL 输出目录，不改变 FE 网络结构。

In [5]:
fe_checkpoint_path = fe_run_dir / "fe_params.msgpack"
if not fe_checkpoint_path.exists():
    raise FileNotFoundError(f"找不到 FE checkpoint：{fe_checkpoint_path}")

fe_state_for_ol, _ = create_fe_state(
    cfg,
    jax.random.PRNGKey(cfg.seed + 30_000),
)
loaded_fe_params = serialization.from_bytes(
    fe_state_for_ol.params,
    fe_checkpoint_path.read_bytes(),
)
fe_state_for_ol = fe_state_for_ol.replace(params=loaded_fe_params)
normalizer_for_ol = load_normalizer(fe_run_dir)

print("Loaded FE checkpoint:", fe_checkpoint_path)
print(
    "Normalizer:",
    f"std_P={float(normalizer_for_ol.std_p):.6e}",
    f"mean_f={float(normalizer_for_ol.mean_f):.6e}",
    f"std_f={float(normalizer_for_ol.std_f):.6e}",
)

E0712 21:08:19.868045 3608243 xtile_compiler.cc:399] Fusion: gemm_fusion_dot_general.1 = f32[4096,512]{1,0} fusion(args_0_.1, args_1_.1), kind=kCustom, calls=gemm_fusion_dot_general.1_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"4","output_tiles":[{"sizes":["64","64"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0712 21:08:19.868129 3608243 xtile_compiler.cc:401] Computation: gemm_fusion_dot_general.1_computation.clone {
  parameter_0 = f32[4096,515]{1,0} parameter(0)
  parameter_1 = f32[515,512]{1,0} parameter(1)
  ROOT dot_general.0 = f32[4096,512]{1,0} dot(parameter_0, parameter_1), lhs_contracting_dims={1}, rhs_contracting_dims={0}, backend_config={"sizes":["64"]}
}
E0712 21:08:19.882532 3608238 x

Loaded FE checkpoint: /home/user/data/Hollon/海洋工程水动力/polar_annulus_sno_code/out_polar_annulus_sno/polar_v1/fe_params.msgpack
Normalizer: std_P=2.718920e-01 mean_f=-2.725592e-03 std_f=1.427547e+02


## 6. 训练前检查一个新 batch

这里仅验证采样结果的形状和有限性。正式训练时，`train_operator` 会在每一步重新生成 batch。

In [6]:
check_batch = sample_batch(
    jax.random.PRNGKey(ol_cfg.seed + 40_000),
    ol_cfg,
)

for name, value in check_batch._asdict().items():
    assert bool(jnp.all(jnp.isfinite(value))), f"{name} contains NaN/Inf"
    print(f"{name:16s}: {value.shape}")

boundary_coords : (768, 128, 2)
boundary_flux   : (768, 128)
pod_coords      : (4096, 2)
probe_coords    : (1024, 2)
p_pod           : (768, 4096)
f_pod           : (768, 4096)
p_probe         : (768, 1024)
f_probe         : (768, 1024)
k_values        : (768,)


## 7. 训练 Operator Learner

运行后将显示 `tqdm` 进度条。每到日志间隔，会在独立的新评估 batch 上打印：

- `train_MSE`：训练 batch 潜空间 MSE；
- `eval_MSE`：独立 batch 潜空间 MSE；
- `RL2(z_P)`：潜空间相对 L2；
- `RL2(P@probe)`：解码到随机 probe 点后的物理压力相对 L2。

本单元每次执行都会从随机初始化重新开始 OL 训练。

In [ ]:
ol_state = train_operator(
    ol_cfg,
    fe_state_for_ol,
    normalizer_for_ol,
)

E0712 21:08:26.819737 3608249 xtile_compiler.cc:399] Fusion: gemm_fusion_dot.1 = f32[8,65,65]{2,1,0} fusion(bitcast.11, bitcast.10), kind=kCustom, calls=gemm_fusion_dot.1_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"8","output_tiles":[{"sizes":["1","128","256"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0712 21:08:26.819818 3608249 xtile_compiler.cc:401] Computation: gemm_fusion_dot.1_computation.clone {
  parameter_0 = f32[65,8,64]{2,1,0} parameter(0)
  transpose.4 = f32[8,65,64]{2,1,0} transpose(parameter_0), dimensions={1,0,2}
  parameter_1 = f32[65,8,64]{2,1,0} parameter(1)
  transpose.5 = f32[8,64,65]{2,1,0} transpose(parameter_1), dimensions={1,2,0}
  ROOT dot.2 = f32[8,65,65]{2,1,0} dot(trans

[OL eval] batch=96, probe_points=1024


Operator training:   0%|                                                                                      …

E0712 21:08:52.226136 3608248 xtile_compiler.cc:399] Fusion: gemm_fusion_dot.111 = f32[8,2048,512]{2,1,0} fusion(bitcast.3, bitcast.2), kind=kCustom, calls=gemm_fusion_dot.111_computation.clone, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"fusion_backend_config":{"kind":"__triton_nested_gemm_fusion","block_level_fusion_config":{"num_warps":"8","output_tiles":[{"sizes":["1","128","128"]}],"num_ctas":1,"num_stages":4,"is_tma_allowed":false,"is_warp_specialization_allowed":false}},"force_earliest_schedule":false,"reification_cost":[],"device_type":"DEVICE_TYPE_INVALID"}
E0712 21:08:52.226204 3608248 xtile_compiler.cc:401] Computation: gemm_fusion_dot.111_computation.clone {
  parameter_0.86 = f32[8,3120,2048]{2,1,0} parameter(0)
  parameter_1.86 = f32[512,8,3120]{2,1,0} parameter(1)
  ROOT dot = f32[8,2048,512]{2,1,0} dot(parameter_0.86, parameter_1.86), lhs_batch_dims={0}, lhs_contracting_dims={1}, rhs_batch_dims={1}, rhs_contracting_dims={2}, backend_config={"

[OL 0000001] train_MSE=9.6518e-01 eval_MSE=1.2569e+00 RL2(z_P)=3.6043e+00 RL2(P@probe)=2.3666e+00


## 8. 读取并可视化训练历史

历史文件在每次日志输出后都会更新，因此训练中断后也可以读取已经保存的部分。

In [ ]:
history_npz_path = ol_cfg.output_dir / "operator_training_history.npz"
history_csv_path = ol_cfg.output_dir / "operator_training_history.csv"

if not history_npz_path.exists():
    raise FileNotFoundError(f"找不到训练历史：{history_npz_path}")

with np.load(history_npz_path) as archive:
    history = {name: archive[name].copy() for name in archive.files}

steps = history["step"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), constrained_layout=True)

axes[0].semilogy(
    steps,
    history["train_latent_mse"],
    label="train latent MSE",
    linewidth=1.6,
)
axes[0].semilogy(
    steps,
    history["eval_latent_mse"],
    label="eval latent MSE",
    linewidth=1.6,
)
axes[0].set_xlabel("step")
axes[0].set_ylabel("MSE")
axes[0].set_title("Latent-space loss")
axes[0].grid(True, which="both", alpha=0.25)
axes[0].legend(frameon=False)

axes[1].semilogy(
    steps,
    history["eval_latent_relative_l2"],
    label=r"RL2($z_P$)",
    linewidth=1.6,
)
axes[1].semilogy(
    steps,
    history["eval_physical_relative_l2"],
    label=r"RL2($P$ at probes)",
    linewidth=1.6,
)
axes[1].set_xlabel("step")
axes[1].set_ylabel("relative L2")
axes[1].set_title("Independent evaluation batch")
axes[1].grid(True, which="both", alpha=0.25)
axes[1].legend(frameon=False)

curve_path = ol_cfg.output_dir / "operator_training_curves.png"
fig.savefig(curve_path, dpi=200, bbox_inches="tight")
plt.show()

print("History CSV:", history_csv_path)
print("History NPZ:", history_npz_path)
print("Training curves:", curve_path)
print("Final OL checkpoint:", ol_cfg.output_dir / "ol_params.msgpack")
print("Latest OL checkpoint:", ol_cfg.output_dir / "ol_params_latest.msgpack")

## 9. 正式训练步骤

1. 保持 `OL_RUN_MODE = "smoke"`，完整运行 Notebook，确认 5 步训练和日志文件正常。
2. 将 `OL_RUN_MODE` 改为 `"full"`。
3. 重启内核并按顺序重新运行全部单元。
4. 长时间训练期间，可在另一个 Notebook 中反复读取 `operator_training_history.npz` 绘图；文件采用原子替换，不会读到半写入状态。

注意：当前实现不自动续训。再次运行训练单元会重新初始化 OL；已有 checkpoint 会被后续保存覆盖。

显存说明：评估默认使用独立小批量（每个尺度 16 个样本）和 256 个探针点，不会复制完整训练批次。如果服务器显存仍不足，可先把 `ol_eval_sample_size` 降到 4、把 `ol_eval_probe_points` 降到 64；这只影响监控指标的统计精度，不改变训练批量。